In [1]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# FIXED: imports animal shelter data from CRUD module
from CRUD_Python_Module import AnimalShelter



###########################
# Data Manipulation / Model
###########################

# FIXED: hardcoded credentials
db = AnimalShelter()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))


## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# FIXED: Adds in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    # FIXED: places image inside layout, shrinks image for better layout
    html.Center(html.A(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),style={'width':'200px','height':'auto','display':'block'}),href='https://snhu.edu')),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    # FIXED: custom name and date header
    html.Center(html.H2("Summer Stinnett - Enhanced November 23, 2026")),
    
    html.Hr(),
    html.Div(
        
# FIXED: adds interactive filtering options
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'All Animals', 'value': 'reset'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster/Tracking', 'value': 'disaster'}
            ],
            value='reset',
            # displays labels side by side instead of top down
            labelStyle={'display':'inline-block'}
        ),

    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         # FIXED: table styling
                         # 10 results per page
                         page_size=10,
                         # automatic handling or sorting and filtering
                         sort_action="native",
                         filter_action="native",
                         # only one row selected at a time, starts program with first row selected
                         row_selectable="single",
                         selected_rows=[0],
                         # table styling
                         style_table={'overflowX':'auto'},
                         style_cell={'textAlign':'left','minWidth':'120px','width':'120px','maxWidth':'180px','whiteSpace':'normal'},

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    # The query for each rescue type now lives in the data layer.
    return db.read(db.rescue_query(filter_type))

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('filter-type', 'value'),
     Input('datatable-id', 'filter_query'),
     Input('datatable-id', 'derived_virtual_data')])
def update_graphs(filter_type, filter_query, viewData):
    if filter_query:
        # Typed column filters are active: count what the table is showing
        if not viewData:
            return []
        counts = pd.DataFrame(viewData)['breed'].value_counts()
        rows = [{'breed': b, 'count': int(c)} for b, c in counts.head(10).items()]
        rest = int(counts.iloc[10:].sum())
        if rest:
            rows.append({'breed': 'Other', 'count': rest})
    else:
        # No typed filters: let MongoDB do the counting
        rows = db.breed_counts(db.rescue_query(filter_type))
    if not rows:
        return []
    return [dcc.Graph(figure=px.pie(pd.DataFrame(rows), names='breed', values='count', title='Animal Breeds Distribution'))]

    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if not viewData or not index:
        return
    row = pd.DataFrame.from_dict(viewData).iloc[index[0]]
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[row['location_lat'], row['location_long']], children=[
                dl.Tooltip(row['breed']),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(row['name'])
                ])
            ])
        ])
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run(jupyter_mode="external", debug=False)

Dash app running on http://127.0.0.1:8050/


C:\Users\summe\cs340-venv\Lib\site-packages\jupyter_dash\jupyter_app.py:103: UserWarning: JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.
  super(JupyterDash, self).__init__(name=name, **kwargs)
